# Feature Engineering — Dữ liệu Ngoài (External Features)

Notebook này xây dựng các feature từ **nguồn dữ liệu ngoài** bổ sung cho bộ dữ liệu bán lẻ Ecuador.

## Các Feature được tạo ra

### Oil Price Features
| Feature | Mô tả |
|---|---|
| `oil_price` | Giá dầu thô WTI đã được ffill (không còn NaN cuối tuần) |
| `oil_price_lag_7` | Giá dầu 1 tuần trước |
| `oil_price_lag_14` | Giá dầu 2 tuần trước (lag có correlation tốt nhất) |
| `oil_price_rolling_mean_28` | Trung bình động 28 ngày — nắm bắt xu hướng dài hạn |
| `oil_price_change_pct` | % thay đổi so với tuần trước — đo lường "shock" giá |

### Store Encoding Features
| Feature | Mô tả |
|---|---|
| `cluster` | Cluster số của store (1–17, dùng trực tiếp) |
| `store_type_encoded` | Label encoding: A→0, B→1, C→2, D→3, E→4 |
| `city_freq` | Frequency encoding của thành phố (tỷ lệ số store trong thành phố đó) |
| `state_freq` | Frequency encoding của tỉnh/bang |

## Tổng quan Pipeline
```
Load Data → Oil: ffill → Lag 7/14 → Rolling 28 → Pct Change → Merge
         → Store: Label Encode type → Freq Encode city/state → Merge
         → Output
```

## 1. Import Thư viện

In [1]:
import pandas as pd
import numpy as np

## 2. Đọc Dữ liệu (Load Data)

Pipeline cần ba bộ dữ liệu:
- **`train_cleaned.csv`** — lịch sử doanh số chính.
- **`cleaned_oil.csv`** — giá dầu thô WTI theo ngày (có thể thiếu ngày cuối tuần/lễ).
- **`stores_cleaned.csv`** — thông tin store: loại, cluster, thành phố, tỉnh.

In [2]:
BASE_PATH = r'D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data'

df_train  = pd.read_csv(rf'{BASE_PATH}\processed\train_cleaned.csv')
df_oil    = pd.read_csv(rf'{BASE_PATH}\processed\cleaned_oil.csv')
df_stores = pd.read_csv(rf'{BASE_PATH}\processed\stores_cleaned.csv')

print(f'Train shape  : {df_train.shape}')
print(f'Oil shape    : {df_oil.shape}')
print(f'Stores shape : {df_stores.shape}')
df_oil.head(3)

Train shape  : (3000888, 6)
Oil shape    : (1218, 2)
Stores shape : (54, 6)


,date,dcoilwtico
0,2013-01-01,93.14
1,2013-01-02,93.14
2,2013-01-03,92.97


---
## Phần 1 — Oil Price Features

Giá dầu là yếu tố kinh tế vĩ mô quan trọng nhất ảnh hưởng đến doanh số bán lẻ tại Ecuador — một nước xuất khẩu dầu mỏ. Khi giá dầu tăng, thu nhập quốc gia tăng, thúc đẩy tiêu dùng; khi giá dầu giảm mạnh, sức mua co lại.

In [3]:
def add_oil_features(
    df: pd.DataFrame,
    oil_df: pd.DataFrame,
    date_col: str = "date",
) -> pd.DataFrame:
   
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Bước 1: Chuẩn bị oil_df — tạo một bảng giá dầu hoàn chỉnh theo ngày
    oil = oil_df.copy()
    oil = oil.rename(columns={"dcoilwtico": "oil_price"})
    oil["date"] = pd.to_datetime(oil["date"])
    oil = oil[["date", "oil_price"]].sort_values("date").reset_index(drop=True)

    # Tạo dải ngày liên tục từ ngày đầu đến ngày cuối trong dữ liệu dầu
    # (để không bỏ sót ngày cuối tuần/ngày lễ khi tính rolling/lag)
    full_date_range = pd.DataFrame({
        "date": pd.date_range(start=oil["date"].min(), end=oil["date"].max(), freq="D")
    })
    oil = full_date_range.merge(oil, on="date", how="left")

    # Bước 2: Xử lý Missing Values bằng Forward Fill (ffill)
    # Lý do dùng ffill: cuối tuần/ngày lễ không có giá dầu mới,
    # nhưng giá dầu ngày hôm qua vẫn là thông tin tốt nhất ta có.
    oil["oil_price"] = oil["oil_price"].ffill()

    # Backward fill cho trường hợp thiếu ở đầu chuỗi (hiếm)
    oil["oil_price"] = oil["oil_price"].bfill()

    # Bước 3: Tạo Lag Features
    # Tại sao cần lag?
    # → Tác động kinh tế của giá dầu không xảy ra ngay lập tức.
    #   Khi giá dầu tăng, người dân & chính phủ mất vài tuần mới điều chỉnh
    #   hành vi chi tiêu. EDA cho thấy lag 14 ngày có correlation tốt nhất.

    oil["oil_price_lag_7"]  = oil["oil_price"].shift(7)   # 1 tuần trước
    oil["oil_price_lag_14"] = oil["oil_price"].shift(14)  # 2 tuần trước (best lag)

    # Bước 4: Rolling Mean 28 ngày
    # Tại sao cần rolling mean?
    # → Giá dầu dao động hàng ngày, rolling mean 28 ngày "làm mịn" 
    #   nhiễu ngắn hạn và nắm bắt xu hướng dài hơn (bull/bear market).
    # min_periods=1: tính trung bình dù chưa đủ 28 ngày (tránh NaN đầu chuỗi)

    oil["oil_price_rolling_mean_28"] = (
        oil["oil_price"].rolling(window=28, min_periods=1).mean()
    )

    # Bước 5: Percentage Change (% thay đổi so với tuần trước)
    # Tại sao cần oil_price_change_pct?
    # → Capture được "shock bất ngờ": giá dầu tăng 10% trong 1 tuần 
    #   có tác động tâm lý khác hẳn mức tăng nhỏ từng ngày.
    # pct_change(7): so sánh với cùng ngày tuần trước

    oil["oil_price_change_pct"] = oil["oil_price"].pct_change(periods=7)

    # Bước 6: Merge vào DataFrame chính
    oil_cols = [
        "date",
        "oil_price",
        "oil_price_lag_7",
        "oil_price_lag_14",
        "oil_price_rolling_mean_28",
        "oil_price_change_pct",
    ]
    df = df.merge(oil[oil_cols], on="date", how="left")

    return df

---
## Phần 2 — Store & Product Encoding

Mỗi store có đặc trưng riêng (loại, vị trí địa lý, cluster) ảnh hưởng đến mức doanh số nền. Ta encode các đặc trưng này thành dạng số để mô hình có thể học được sự khác biệt giữa các store.

In [4]:
def add_store_encoding(
    df: pd.DataFrame,
    stores_df: pd.DataFrame,
) -> pd.DataFrame:

    df = df.copy()
    stores = stores_df.copy()

    # Bước 1: Label Encoding cho store_type (A, B, C, D, E → 0, 1, 2, 3, 4)
    # store_type chỉ có 5 loại duy nhất → Label Encoding đơn giản là đủ.
    # Chuyển A→0, B→1, ... theo thứ tự alphabet (nhất quán, không ngẫu nhiên).
    type_mapping = {t: i for i, t in enumerate(sorted(stores["type"].unique()))}
    stores["store_type_encoded"] = stores["type"].map(type_mapping)

    # Bước 2: Frequency Encoding cho city và state
    # Frequency encoding: thay tên thành phố/tỉnh bằng xác suất xuất hiện
    # Các thành phố lớn có nhiều cửa hàng → giá trị freq cao hơn
    # → model hiểu được quy mô địa lý một cách ngầm định

    city_freq  = stores["city"].value_counts(normalize=True).to_dict()
    state_freq = stores["state"].value_counts(normalize=True).to_dict()

    stores["city_freq"]  = stores["city"].map(city_freq)
    stores["state_freq"] = stores["state"].map(state_freq)

    # Bước 3: Giữ cluster nguyên (đã là số)
    # cluster là số từ 1-17, LightGBM xử lý trực tiếp được.
    # Không cần encode thêm.

    # Bước 4: Merge vào DataFrame chính theo store_nbr
    store_cols = [
        "store_nbr",
        "cluster",            # giữ nguyên
        "store_type_encoded", # A-E → 0-4
        "city_freq",          # frequency của thành phố
        "state_freq",         # frequency của tỉnh/bang
    ]
    df = df.merge(stores[store_cols], on="store_nbr", how="left")

    return df

---
## Phần 3 — Pipeline Tổng hợp

Gộp hai hàm trên thành một pipeline duy nhất để tiện tái sử dụng.

In [5]:
def build_external_features(
    df: pd.DataFrame,
    oil_df: pd.DataFrame,
    stores_df: pd.DataFrame,
    date_col: str = "date",
) -> pd.DataFrame:

    df = add_oil_features(df, oil_df, date_col=date_col)
    df = add_store_encoding(df, stores_df)
    return df

---
## Phần 4 — Danh sách tên Feature

Ba danh sách hằng số để dùng ở bước model training: chọn feature, xác thực pipeline, và ghi chép.

In [6]:
OIL_FEATURE_NAMES = [
    "oil_price",
    "oil_price_lag_7",
    "oil_price_lag_14",
    "oil_price_rolling_mean_28",
    "oil_price_change_pct",
]

STORE_FEATURE_NAMES = [
    "cluster",
    "store_type_encoded",
    "city_freq",
    "state_freq",
]

ALL_EXTERNAL_FEATURE_NAMES = OIL_FEATURE_NAMES + STORE_FEATURE_NAMES

print(f'Oil features   : {OIL_FEATURE_NAMES}')
print(f'Store features : {STORE_FEATURE_NAMES}')
print(f'Total          : {len(ALL_EXTERNAL_FEATURE_NAMES)} features')

Oil features   : ['oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct']
Store features : ['cluster', 'store_type_encoded', 'city_freq', 'state_freq']
Total          : 9 features


---
## Phần 5 — Chạy Pipeline trên Dữ liệu Thực

Áp dụng toàn bộ pipeline lên `train_cleaned.csv` và kiểm tra kết quả.

In [7]:
df_featured = build_external_features(df_train, df_oil, df_stores)

print(f'Input shape  : {df_train.shape}')
print(f'Output shape : {df_featured.shape}')
print(f'New columns  : {df_featured.shape[1] - df_train.shape[1]}')
print()
print('=== Feature Summary ===')
df_featured[ALL_EXTERNAL_FEATURE_NAMES].describe().T[['mean', 'min', 'max']]

Input shape  : (3000888, 6)
Output shape : (3000888, 15)
New columns  : 9

=== Feature Summary ===


,mean,min,max
oil_price,67.924899,26.190000,110.620000
oil_price_lag_7,68.009296,26.190000,110.620000
oil_price_lag_14,68.083719,26.190000,110.620000
oil_price_rolling_mean_28,68.279963,30.210000,108.076429
oil_price_change_pct,-0.001698,-0.171989,0.287284
cluster,8.481481,1.000000,17.000000
store_type_encoded,2.000000,0.000000,4.000000
city_freq,0.149520,0.018519,0.333333
state_freq,0.182442,0.018519,0.351852


---
## Phần 6 — Smoke Test với Dữ liệu Giả

Kiểm tra pipeline trên bộ dữ liệu tổng hợp nhỏ để xác nhận:
- Tất cả feature đều có mặt (`✓` / `✗ MISSING`).
- `oil_price` không còn NaN sau ffill.
- `city_freq` nằm trong khoảng `[0, 1]`.
- `oil_price_lag_14` có giá trị hợp lệ sau 14 ngày warmup.

In [8]:
print("=" * 60)
print("SMOKE TEST: feature_external")
print("=" * 60)

# --- Tạo data giả để test ---
np.random.seed(42)
dates = pd.date_range("2016-01-01", periods=90, freq="D")

# DataFrame chính (train giả)
mock_train = pd.DataFrame({
    "date":      list(dates) * 3,
    "store_nbr": [1] * 90 + [2] * 90 + [3] * 90,
    "family":    ["GROCERY I"] * 270,
    "sales":     np.random.rand(270) * 1000,
})

# Oil DataFrame giả (thêm vài NaN để test ffill)
mock_oil = pd.DataFrame({
    "date":        pd.date_range("2015-12-01", periods=120, freq="D"),
    "dcoilwtico":  np.where(
        np.random.rand(120) < 0.15,  # 15% ngày thiếu giá (cuối tuần/lễ)
        np.nan,
        50 + np.random.randn(120) * 5  # giá dầu ~50 USD
    ),
})

# Stores DataFrame giả
mock_stores = pd.DataFrame({
    "store_nbr": [1, 2, 3],
    "type":      ["A", "B", "A"],
    "cluster":   [1, 3, 1],
    "city":      ["Quito", "Guayaquil", "Quito"],
    "state":     ["Pichincha", "Guayas", "Pichincha"],
})

# --- Chạy pipeline ---
result = build_external_features(mock_train, mock_oil, mock_stores)

print(f"\nShape đầu vào:  {mock_train.shape}")
print(f"Shape đầu ra:   {result.shape}")

print("\n--- Kiểm tra từng feature ---")
all_ok = True
for col in ALL_EXTERNAL_FEATURE_NAMES:
    if col in result.columns:
        null_pct = result[col].isna().mean() * 100
        sample_val = result[col].dropna().iloc[0] if not result[col].dropna().empty else "N/A"
        print(f"  ✓  {col:<35}  NaN={null_pct:5.1f}%  sample={sample_val:.4f}" if isinstance(sample_val, float) else f"  ✓  {col:<35}  NaN={null_pct:5.1f}%  sample={sample_val}")
    else:
        print(f"  ✗  {col:<35}  MISSING!")
        all_ok = False

# --- Kiểm tra oil_price không còn NaN sau ffill ---
oil_nan = result["oil_price"].isna().sum()
print(f"\nSau ffill, oil_price NaN còn lại: {oil_nan} (phải = 0 hoặc rất nhỏ)")

# --- Kiểm tra store_type_encoded ---
unique_types = result["store_type_encoded"].unique()
print(f"store_type_encoded unique values: {sorted(unique_types)}")

# --- Kiểm tra city_freq trong khoảng [0, 1] ---
assert result["city_freq"].dropna().between(0, 1).all(), "city_freq ngoài khoảng [0,1]!"
print("city_freq trong khoảng [0, 1]: ✓")

# --- Kiểm tra lag_14 có giá trị sau 14 ngày đầu ---
lag14_after_warmup = result[result["date"] >= dates[14]]["oil_price_lag_14"]
nan_after = lag14_after_warmup.isna().sum()
print(f"oil_price_lag_14 NaN sau warmup 14 ngày: {nan_after} (nên thấp)")

print("\n" + ("✅ TẤT CẢ TESTS PASSED!" if all_ok else "❌ CÓ LỖI, kiểm tra lại!"))
print("\n--- Xem trước 3 dòng đầu ---")
print(result[["date", "store_nbr"] + ALL_EXTERNAL_FEATURE_NAMES].head(3).to_string())

SMOKE TEST: feature_external

Shape đầu vào:  (270, 4)
Shape đầu ra:   (270, 13)

--- Kiểm tra từng feature ---
  ✓  oil_price                            NaN=  1.1%  sample=49.2647
  ✓  oil_price_lag_7                      NaN=  1.1%  sample=53.9583
  ✓  oil_price_lag_14                     NaN=  1.1%  sample=52.9758
  ✓  oil_price_rolling_mean_28            NaN=  1.1%  sample=52.6599
  ✓  oil_price_change_pct                 NaN=  1.1%  sample=-0.0870
  ✓  cluster                              NaN=  0.0%  sample=1
  ✓  store_type_encoded                   NaN=  0.0%  sample=0
  ✓  city_freq                            NaN=  0.0%  sample=0.6667
  ✓  state_freq                           NaN=  0.0%  sample=0.6667

Sau ffill, oil_price NaN còn lại: 3 (phải = 0 hoặc rất nhỏ)
store_type_encoded unique values: [np.int64(0), np.int64(1)]
city_freq trong khoảng [0, 1]: ✓
oil_price_lag_14 NaN sau warmup 14 ngày: 3 (nên thấp)

✅ TẤT CẢ TESTS PASSED!

--- Xem trước 3 dòng đầu ---
        date  stor

---
## Lưu Kết quả (Save Output)

Lưu dataframe đã được bổ sung external features để các notebook modelling downstream sử dụng.

In [9]:
OUTPUT_PATH = rf'{BASE_PATH}\processed\train_external_features.csv'
df_featured.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to : {OUTPUT_PATH}')
print(f'Shape    : {df_featured.shape}')
print(f'\nFinal columns:')
print(df_featured.columns.tolist())

Saved to : D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\train_external_features.csv
Shape    : (3000888, 15)

Final columns:
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct', 'cluster', 'store_type_encoded', 'city_freq', 'state_freq']
